# <center> INTRODUCTION À PYTHON POUR L'ÉCONOMIE EMPIRIQUE</center>
## <center> COURS 2: </center>
## <center> LECTURE ET ANALYSE DE BASIQUE DE DONNÉES </center>
#### <center>Michal Urdanivia (UGA)</center>
#### <center> michal.wong-urdanivia@univ-grenoble-alpes.fr </center>

### <center>1. PRÉSENTATION </center>

Nous allons dans ce notebook nous servir de données de AirBnb pour la ville de Oslo, lesquelles sont disponibles à **Inside AirBnb**: 

http://insideairbnb.com/get-the-data.html

Un dictionnaire des variables disponibles est à consulter [ici](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit#gid=982310896).

Les données que nous employons concernent:

- les listings,
- les prix.

In [ ]:
# Importation de bibliothèques utilisées

import numpy as np
import pandas as pd

### <center>2. LECTURE DES DONNÉES </center>

La bibliothèque Pandas contient plusieurs fonction pour lire des données qui se distinguent par le format dans lequel les données se présentent. Par exemple, notons les fonction suivantes,

- `pd.read_csv()`
- `pd.read_html()`
- `pd.read_parquet()`

Pour ce qui va suivre, on peut noter `pd.read_csv()` peut importer des données depuis le web.

La première base de données que nous importons contient les listings de AirBnb pour la ville de Oslo.

In [ ]:
url_listings = "https://data.insideairbnb.com/norway/oslo/oslo/2025-09-29/visualisations/listings.csv"
df_listings = pd.read_csv(url_listings)

La deuxième base de données contient des informations sur les montants(prix) des transactions et le calendrier correspondant. Ces données se présentent en format compressé. Pour les importer directement on utilise ici l'option  `compression`.
 

In [ ]:
url_prices = "https://data.insideairbnb.com/norway/oslo/oslo/2025-09-29/data/calendar.csv.gz"
df_prices = pd.read_csv(url_prices, compression="gzip")

### <center>3. EXPLORATION BASIQUE DES DONNÉES </center>

> Méthodes
>
>- `info()`
>- `head()`
>- `describe()`

L'appel de' `info()` data frame pandas donne des informations sur les données dans leur ensemble et les variables présentes.

In [ ]:
df_listings.info()

Avec l'option `verbose=False` on peut se contenter d'informations sur la dimension des données.

In [ ]:
df_listings.info(verbose=False)

On peut aussi afficher des lignes de la base afin de voir à quoi cela ressemble. Pour cela on peut utiliser la méthode `head()`  qui par défaut affiche les 5 premières lignes. Avec `head(nbr)` ou `head(n = nbr)`où `nbr` est un entier positif on affiche les `nbr` premières lignes.

In [ ]:
df_listings.head()

La méthode `describe()` permet d'afficher une description des données. Si plusieurs variables sont présentes il est préférable d'afficher une transposée du tableau de sortie par défaut en utilisant l'attribut `.T`.

In [ ]:
df_listings.describe().T[:5]

On peut choisir les variables à afficher avec l'option `include` avec `include='all'`
pour inclure aussi les variables catégorielles.

In [ ]:
df_listings.describe(include='all').T[:5]

L'attribut `.columns` donne une liste des colonnes dans la base de données.

In [ ]:
df_listings.columns

Pour obtenir l'indice nous employons l'attribut `.index`,

In [ ]:
df_listings.index

### <center>4. SÉLECTION DE DONNÉES </center>

Pour accéder à une seule colonne du data frame on utilise le nom de cette colonne comme la clé d'un élément dans un objet de type dictionnaire.

In [ ]:
df_listings['price']

L'attribut `.iloc` permet la sélection de lignes et colonnes à partir de l'indice,

In [ ]:
df_listings.iloc[:7, 5:9]

Pour ne sélectionner que des lignes ou colonnes on utilise `:` sur la dimension(lignes ou colonnes) où la sélection ne s'applique pas.

In [ ]:
df_listings.iloc[:, 5:9].head()

Quant à l'attribut `.loc` il permet d'utiliser les noms des lignes et colonnes.

In [ ]:
df_listings.loc[:, ['neighbourhood', 'latitude', 'longitude']].head()

Et on peut aussi sélectionner des rangs.

In [ ]:
df_listings.loc[:, 'neighbourhood':'room_type'].head()

Une manière rapide de sélectionner des **colonnes numériques** est la fonction
`.select_dtypes()` fonction.

In [ ]:
df_listings.select_dtypes(include=['number']).head()

Les autres types sont:

- `object` pour les chaînes de caractère,
- `bool` pour les booléens,
- `int` pour les entiers,
- `float` pour les floats(par abus nous dirons qu'il s'agit des nombres réels)

On peut aussi sélectionner des lignes à partir d'un opérateur logique,

In [ ]:
df_listings.loc[df_listings['number_of_reviews']>500, :].head()

On peut aussi utiliser des conjonctions logiques mais il faut se souvenir d'utiliser des parenthèses

**Remarque**: les expressions `and` et  `or` expressions ne fonctionnent pas dans ce cadre. On doit utiliser `&` et `|`.

In [ ]:
df_listings.loc[(df_listings['number_of_reviews']>300) &
                (df_listings['reviews_per_month']>7), 
                :].head()

Pour une seule colonne(i.e., un objet de type Series) on peut obtenir les valeurs prises on utilisant la fonction `unique()`.

In [ ]:
df_listings['neighbourhood'].unique()

Pour plusieurs colonnes, on peut utiliser la fonction `drop_duplicates`.

In [ ]:
df_listings[['neighbourhood', 'room_type']].drop_duplicates()

### <center>5. AGGREGATION ET TABLEAUX CROISÉS </center>

Pour calculer des statistiques par groupe on peut utiliser la fonction `.groupby()`.

In [ ]:
df_listings.groupby('neighbourhood')[['price', 'reviews_per_month']].mean()

Si l'on souhaite appliquer plus d'une fonction et notamment sur plusieurs colonnes, on peut utiliser `.aggregate()` qui est abrégée en  `.agg()`. Son argument est un dictionnaire où les variables sont les clés et les valeurs sont des listes de fonctions.


In [ ]:
df_listings.groupby('neighbourhood').agg({"reviews_per_month": ["mean"],
                                          "price": ["min", np.max]}).reset_index()

Le problème avec cette syntaxe est qu'elle génère une structure hiérarchique pour les noms des variables, avec laquelle il peut être difficile de travailler. Dans l'exemple précédent pour accéder au prix moyen on doit utiliser `df.price["min"]`.

Pour faire simultanément l'agrégation et renommer les variables, on peut utiliser la syntaxe:  `agg(output_var = ("input_var", function))`.

In [ ]:
df_listings.groupby('neighbourhood').agg(mean_reviews=("reviews_per_month", "mean"),
                                         min_price=("price", "min"),
                                         max_price=("price", np.max)).reset_index()

Pour faire de tableaux croisés on utilise la fonction `.pivot_table()` dont les arguments sont:
- `index`: lignes,
- `columns`: colonnes,
- `values`: valeurs
- `aggfunc`: fonction d'agrégation.

In [ ]:
df_listings.pivot_table(index='neighbourhood', columns='room_type', values='price', aggfunc='mean')